# Pipeline walkthrough

Every stage of README section 2, with its intermediates exposed. Section 1.2
requires the intermediates to be inspectable, and this is what that buys you.

In [ ]:
import sys
from pathlib import Path

# Run from anywhere: notebooks/ is a sibling of src/.
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

In [ ]:
from tfidf_stability.preprocessing.pipeline import PreprocessingPipeline

pipeline = PreprocessingPipeline()
text = "The Quick, Brown Foxes were jumping over 2 lazy dogs!"

print("raw        ", text)
print("features   ", pipeline.preprocess(text))

Note the bigrams, joined with `\x1f` (unit separator) -- a character that
cannot occur in normalised text, so a bigram can never collide with a unigram.

In [ ]:
from tfidf_stability.utils.io import read_jsonl
from tfidf_stability.vectorisation.tfidf import TfidfVectoriser

records = list(read_jsonl(REPO / "tests" / "fixtures" / "mini_corpus.jsonl"))
features = [pipeline.preprocess(r["text"]) for r in records]
model = TfidfVectoriser().fit(features, [r["doc_id"] for r in records])

print(f"documents {model.n_documents}   vocabulary {model.n_features}")
print(f"reduction {model.reduction}   log {model.idf.log_impl}")

## IDF is not computed with `math.log`

`math.log` is not correctly rounded: it differs from the correctly-rounded value
in about 15% of IDF entries, and UCRT, glibc and Apple libm each round
differently. IDF is computed once via `decimal.Decimal.ln()` at 60 digits and
rounded once -- see `docs/spec_addenda.md#g13`.

The cell below shows the disagreement directly.

In [ ]:
import math
from decimal import Decimal, getcontext

getcontext().prec = 60
n = model.n_documents

disagree = 0
for term_id in range(model.n_features):
    token = model.vocabulary.token_of(term_id)
    df = model.vocabulary.df_of(token)
    exact = float(Decimal(1 + n) / Decimal(1 + df))
    platform_log = math.log((1 + n) / (1 + df))
    correct = float(Decimal((1 + n)) .ln() - Decimal((1 + df)).ln())
    if platform_log != correct:
        disagree += 1

print(f"{disagree} of {model.n_features} raw logarithms differ from correctly-rounded")
print("on this tiny corpus; the effect is ~15% of IDF entries at realistic sizes")

## Intermediates for one document

In [ ]:
intermediates = model.intermediates(0)
for key, value in intermediates.items():
    if not isinstance(value, (list, tuple, dict)):
        print(f"{key:14} {value!r}")

vector = model.document(0)
print(f"\nnnz {len(vector.indices)}   norm {model.norms[0]!r} ({float.hex(model.norms[0])})")
for column, weight in list(zip(vector.indices, vector.values))[:8]:
    token = model.vocabulary.token_of(column)
    print(f"  {token:28} idf={model.idf.values[column]:.9f}  w={weight:.9f}")